<a href="https://colab.research.google.com/github/lizarizvi/f1-race-winner-predictor/blob/main/Formula1_GP_winner_predictor_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# My First ML Model: Predict the Next Grand Prix Winner

**Goal:** Predict who will win the upcoming Formula 1 Grand Prix, using only the last 5 full seasons (2021–2025) plus every 2026 race that has already happened.

## The plan
| Step | What we do | Why |
|---|---|---|
| 1 | Import libraries | Get our tools ready |
| 2 | Download race results (2021 -> 2026) | A model learns from past examples |
| 3 | Clean and look at the data | Garbage in = garbage out |
| 4 | Create "recent form" features (recent performance) | A driver's recent form predicts the future |
| 5 | Split into train / test by time | Test the model on races it has never seen |
| 6 | Build a pipeline: Imputer + Ordinal + One-Hot + Random Forest | Turn raw columns into numbers the model can use |
| 7 | Evaluate with the right metric | Know if the model is actually good |
| 8 | Predict the upcoming race | The real goal |

## Step 1: Import libraries

**What it does:** loads the Python tools we will use.

In [1]:
import requests                      # lets Python download data from a website/API
import time                           # lets us pause (sleep) between downloads
import pandas as pd                   # tables of data (DataFrames)
import numpy as np                    # numeric helpers (e.g. np.nan, np.mean)

from sklearn.base import clone                        # makes a fresh, untrained copy of a model
from sklearn.impute import SimpleImputer              # fills missing values
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder   # text -> numbers
from sklearn.compose import ColumnTransformer         # different cleaning per column group
from sklearn.pipeline import Pipeline                 # chains steps together
from sklearn.ensemble import RandomForestClassifier   # our model: many decision trees voting

pd.set_option("display.width", 160)   # make printed tables wider so they don't wrap
pd.set_option("display.max_columns", 30)  # show up to 30 columns when printing a table

## Step 2: Download the race results (2021 -> 2026)

**What it does:** asks the free Jolpica-F1 API (the successor of the old Ergast database) for every race result in each season, and stores one row per driver per race.

In [2]:
BASE = "https://api.jolpi.ca/ergast/f1"      # address of the free F1 data API
SEASONS = list(range(2021, 2027))            # 2021,2022,2023,2024,2025,2026 (range end is exclusive, so 2027)

def get_json(url, params=None):
    # Download a URL and return its content as a Python dictionary
    for attempt in range(6):                                   # try up to 6 times
        r = requests.get(url, params=params, timeout=30)       # ask the server for the data
        if r.status_code == 429:                               # 429 = "slow down, too many requests"
            time.sleep(5 * (attempt + 1))                      # wait 5s, 10s, 15s ... then retry
            continue                                           # go to the next attempt
        r.raise_for_status()                                   # stop with an error if something else went wrong
        return r.json()                                        # success: convert the text into a dictionary
    raise RuntimeError("API kept saying 'too many requests'. Wait a few minutes and run again.")

def get_season_results(season):
    # Download every driver-result of one season and return it as a list of small dictionaries
    rows = []                                                  # we will collect all rows here
    offset = 0                                                 # start from the first result
    while True:                                                # keep going until we break out
        data = get_json(f"{BASE}/{season}/results.json",
                        {"limit": 100, "offset": offset})["MRData"]   # one page of up to 100 results
        total = int(data["total"])                             # how many results exist in the whole season
        for race in data["RaceTable"]["Races"]:                # loop over races in this page
            last = max(int(x["grid"]) for x in race["Results"])# last real grid slot in this race
            for res in race["Results"]:                        # loop over each driver's result in that race
                grid = int(res["grid"])                        # starting grid
                rows.append({
                    "season": int(race["season"]),             # year
                    "round": int(race["round"]),               # race number in that year (1 = first race)
                    "race_name": race["raceName"],             # e.g. "Monaco Grand Prix"
                    "circuit_id": race["Circuit"]["circuitId"],# track name id, e.g. "monaco"
                    "date": race["date"],                      # race date
                    "driver_id": res["Driver"]["driverId"],    # e.g. "max_verstappen"
                    "constructor_id": res["Constructor"]["constructorId"],  # team, e.g. "red_bull"
                    "grid": grid if grid > 0 else last + 1,    # 0 = pit lane start -> just behind the last car on the grid
                    "position": int(res["position"]),          # finishing position (1 = winner)
                    "points": float(res["points"]),            # championship points scored
                })
        offset += 100                                          # next page starts 100 results later
        if offset >= total:                                    # we have downloaded everything
            break                                              # leave the while loop
        time.sleep(0.5)                                        # small pause between pages
    return rows                                                # give back all rows of that season

all_rows = []                                                  # will hold rows of every season
for s in SEASONS:                                              # repeat for each year
    season_rows = get_season_results(s)                        # download that year
    print(s, "->", len(season_rows), "driver results")         # show progress
    all_rows += season_rows                                    # add them to the big list
    time.sleep(0.5)                                            # small pause before the next season

2021 -> 440 driver results
2022 -> 440 driver results
2023 -> 440 driver results
2024 -> 479 driver results
2025 -> 479 driver results
2026 -> 396 driver results


## Step 3: Build the table and look at it

**What it does:** turns the list into a pandas table, fixes data types, creates our target column 'won', and prints a few checks.

- ML models predict a target. Ours is 'won' -> '1' if the driver finished 1st, otherwise '0'. This makes it a classification problem ("will this driver win: yes/no?").
- In this data grid = 0 means "started from the pit lane", which is not a real grid slot. We give the grid number as grid + 1 to them. All pit-lane starters will have same grid number i.e. grid + 1.
- Always look at your data first. It shows problems early.

In [3]:
df = pd.DataFrame(all_rows)                              # list of dictionaries -> table
df["date"] = pd.to_datetime(df["date"])                  # turn the text date into a real date
df = df.sort_values(["season", "round", "position"]).reset_index(drop=True)   # oldest race first
df["race_id"] = df["season"] * 100 + df["round"]         # one unique number per race, e.g. 202607 = 2026 round 7
df["won"] = (df["position"] == 1).astype(int)            # TARGET: 1 if finished first, else 0

print("Table size (rows, columns):", df.shape)           # how big is our data
print("Races per season:")                               # how many races did each year have
print(df.groupby("season")["round"].nunique())
print("\nWins are rare -> share of rows that are a win:", round(df["won"].mean(), 3))  # about 0.05 (1 winner out of ~20)
print("\nMissing values per column:")                    # count empty cells
print(df.isna().sum())
df.head()                                                # show the first 5 rows

Table size (rows, columns): (2674, 12)
Races per season:
season
2021    22
2022    22
2023    22
2024    24
2025    24
2026    18
Name: round, dtype: int64

Wins are rare -> share of rows that are a win: 0.049

Missing values per column:
season            0
round             0
race_name         0
circuit_id        0
date              0
driver_id         0
constructor_id    0
grid              0
position          0
points            0
race_id           0
won               0
dtype: int64


,season,round,race_name,circuit_id,date,driver_id,constructor_id,grid,position,points,race_id,won
0,2021,1,Bahrain Grand Prix,bahrain,2021-03-28,hamilton,mercedes,2,1,25.0,202101,1
1,2021,1,Bahrain Grand Prix,bahrain,2021-03-28,max_verstappen,red_bull,1,2,18.0,202101,0
2,2021,1,Bahrain Grand Prix,bahrain,2021-03-28,bottas,mercedes,3,3,16.0,202101,0
3,2021,1,Bahrain Grand Prix,bahrain,2021-03-28,norris,mclaren,7,4,12.0,202101,0
4,2021,1,Bahrain Grand Prix,bahrain,2021-03-28,perez,red_bull,21,5,10.0,202101,0


## Step 4: Create "recent form" features

**What it does:** adds columns that describe how a driver and a team have been performing recently.

**Why:** The model should not guess from nothing. Before a race, we already know things like "this driver averaged P2 in his last 5 races". Those are strong clues.

**The big trap: data leakage.** We must only use information available before the race. That is why every rolling calculation uses '.shift(1)': it moves values down by one row so that a race's own result is never used to describe itself.

New columns:
| Column | Meaning |
|---|---|
| driver_avg_finish_5 | Driver's average finishing position over his previous 5 races (lower = better) |
| driver_wins_5 | Wins by the driver in his previous 5 races |
| driver_points_5 | Average points of the driver in his previous 5 races |
| team_points_5 | Team's average points per race in the previous 5 races (shows how fast the car is) |

The first race of any driver has no history -> these become NaN -> the SimpleImputer will fill them in Step 6.

For the team column we first add up both drivers' points per race, and then take the rolling average. (If we rolled over single rows, the second driver would see his teammate's result from the same race, which is leakage.)

In [4]:
df = df.sort_values(["race_id", "position"]).reset_index(drop=True)   # make sure rows are in time order

by_driver = df.groupby("driver_id")                       # look at each driver separately

# average finishing position of the previous 5 races
df["driver_avg_finish_5"] = by_driver["position"].transform(
    lambda s: s.shift(1).rolling(5, min_periods=1).mean())
# number of wins in the previous 5 races
df["driver_wins_5"] = by_driver["won"].transform(
    lambda s: s.shift(1).rolling(5, min_periods=1).sum())
# average points in the previous 5 races
df["driver_points_5"] = by_driver["points"].transform(
    lambda s: s.shift(1).rolling(5, min_periods=1).mean())

# team form: first total points of the team in each race
team = (df.groupby(["constructor_id", "race_id"], as_index=False)    # one row per team per race
          .agg(team_points=("points", "sum"))                        # add both drivers' points
          .sort_values(["constructor_id", "race_id"]))               # time order inside each team
team["team_points_5"] = team.groupby("constructor_id")["team_points"].transform(
    lambda s: s.shift(1).rolling(5, min_periods=1).mean())           # average of previous 5 races
df = df.merge(team[["constructor_id", "race_id", "team_points_5"]],
              on=["constructor_id", "race_id"], how="left")           # attach it back to every driver row

df[["race_name", "driver_id", "grid", "position",
    "driver_avg_finish_5", "driver_wins_5", "team_points_5"]].tail(10)   # peek at the newest rows

,race_name,driver_id,grid,position,driver_avg_finish_5,driver_wins_5,team_points_5
2664,United States Grand Prix,bortoleto,21,13,15.8,0.0,1.0
2665,United States Grand Prix,ocon,15,14,12.6,0.0,1.2
2666,United States Grand Prix,stroll,22,15,19.0,0.0,0.8
2667,United States Grand Prix,alonso,12,16,16.6,0.0,0.8
2668,United States Grand Prix,albon,17,17,18.2,0.0,0.2
2669,United States Grand Prix,sainz,16,18,14.6,0.0,0.2
2670,United States Grand Prix,perez,18,19,17.6,0.0,0.0
2671,United States Grand Prix,bottas,19,20,18.8,0.0,0.0
2672,United States Grand Prix,colapinto,20,21,13.4,0.0,2.8
2673,United States Grand Prix,bearman,13,22,13.2,0.0,1.2


## Step 5: Split into training and test data (by time)

**What it does:** keeps the last 6 completed races as a test set and trains on everything before them.

**Why by time and not random?** In real life we predict the future from the past. A random split would let the model peek at races that happened after the ones it is tested on, which makes results look unrealistically good.

**Why only 6 test races?** Using the newest races as the test also checks how the model copes with the 2026 rule changes. We need to be aware that 6 races is a tiny test, so the score will be noisy.

**Features** are the columns the model is allowed to look at, split by how we will encode them:
- num_cols -> numbers (imputer only)
- ord_cols -> driver_id (OrdinalEncoder)
- ohe_cols -> constructor_id, circuit_id (OneHotEncoder)

In [5]:
N_TEST = 6                                               # how many of the latest races to hold out for testing

race_ids = sorted(df["race_id"].unique())                # all race numbers in time order
test_races = race_ids[-N_TEST:]                          # the last N_TEST races
train = df[~df["race_id"].isin(test_races)].copy()       # everything else = training data
test  = df[df["race_id"].isin(test_races)].copy()        # the held-out test set

num_cols = ["grid", "driver_avg_finish_5", "driver_wins_5",
            "driver_points_5", "team_points_5"]          # numeric columns
ord_cols = ["driver_id"]                                 # category -> ordinal numbers
ohe_cols = ["constructor_id", "circuit_id"]              # category -> one-hot 0/1 columns
features = num_cols + ord_cols + ohe_cols                # all inputs together

X_train, y_train = train[features], train["won"]         # X = inputs, y = answer
X_test,  y_test  = test[features],  test["won"]

print("Training rows:", len(train), "| Test rows:", len(test))
print("Test races:", test[["season", "round", "race_name"]].drop_duplicates().to_string(index=False))

Training rows: 2542 | Test rows: 132
Test races:  season  round                      race_name
   2026     13             Italian Grand Prix
   2026     14             Spanish Grand Prix
   2026     15          Azerbaijan Grand Prix
   2026     16 Bahrain Grand Prix in Malaysia
   2026     17           Singapore Grand Prix
   2026     18       United States Grand Prix


## Step 6: Build the pipeline (Imputer + Encoders + Random Forest)

Models only understand numbers with no gaps. Our data has gaps and text. The pipeline fixes this automatically, in the same way, for training and for new races.

**Three preprocessing tools and why each is used**

1. SimpleImputer(strategy="median") -> numeric columns -> Replaces NaN with the column's median.

2. SimpleImputer(strategy="most_frequent") + OrdinalEncoder -> 'driver_id' -> Fill gaps with the most common driver, then turn each driver into a number. ('hamilton->3, ...)

3. OneHotEncoder(handle_unknown="ignore") -> 'constructor_id', 'circuit_id' -> One 0/1 column per team/track. ('is_ferrari', 'is_monaco'...)

**Why Random Forest?** A single Decision Tree is easy to understand but memorises the data. A Random Forest builds hundreds of slightly different trees on random pieces of data and lets them vote, which is far more stable. It needs no feature scaling, handles a mix of features, and works well on small tables like ours.

**Key settings**
- 'n_estimators=300' -> 300 trees.
- 'min_samples_leaf=3' -> each leaf must contain >=3 rows; stops the trees from memorising single races.
- 'class_weight="balanced_subsample"' -> only ~5% of rows are winners. This tells the model "a win is rare but important, pay attention to it", otherwise it would just answer "nobody wins".

In [6]:
numeric_pipe = SimpleImputer(strategy="median")          # fill missing numbers with the median

driver_pipe = Pipeline([                                 # two steps for the driver column
    ("fill", SimpleImputer(strategy="most_frequent")),   # 1) fill missing names with the most common one
    ("code", OrdinalEncoder(handle_unknown="use_encoded_value",
                            unknown_value=-1)),          # 2) name -> number; unseen driver -> -1
])

ohe_pipe = OneHotEncoder(handle_unknown="ignore")        # unseen team/track -> all zeros

preprocess = ColumnTransformer([                         # apply the right tool to the right columns
    ("num", numeric_pipe, num_cols),
    ("driver", driver_pipe, ord_cols),
    ("onehot", ohe_pipe, ohe_cols)
])

model = Pipeline([
    ("prep", preprocess),                                # clean + encode
    ("forest", RandomForestClassifier(
        n_estimators=300,                                # 300 trees
        min_samples_leaf=3,                              # avoid memorising
        class_weight="balanced_subsample",               # winners are rare, so weigh them more
        random_state=42,                                 # reproducible
        n_jobs=-1)),                                     # use all CPU cores
])

model.fit(X_train, y_train)                              # TRAIN: the model learns from past races
print("Model trained.")

Model trained.


## Step 7: Choose the metric (and why NOT MAE)

**MAE** = the average size of the mistakes, 'average(|prediction - actual|)'. It is great for predicting amounts (lap time, price, points).

**Race-winner accuracy** = out of the test races, in how many did the driver we picked actually win? For each race we give every driver a win-probability, "pick the driver with the highest", and check whether he won.

**Why MAE is the wrong metric here?**
Imagine a 20-driver race and a lazy model that says "0% chance for everybody":
- It’s wrong about one driver (the winner) and right about the other 19.
- Its average error is tiny: 1 mistake out of 20 = 0.05.
- Its row-by-row accuracy looks great: 95%.

MAE doesn’t care about ranking drivers within a race. What we really need is “who’s most likely to win?”, not “how close was your number to 0 or 1?”

**Why race-winner accuracy is better?**
- It measures what we truly care about: "did we name the winner?"
- It is easy to explain: "we got 4 of 6 races right".
- A lazy model scores 0%, so it cannot cheat.
- We compare against a simple baseline: "the pole-sitter (grid 1) wins". If our model cannot beat that, it adds no value.

**Note:** F1 is unpredictable (safety cars, rain, strategy) and we only have 6 test races, so each race is worth ~17 percentage points. Treat the score as a rough guide.

In [7]:
test["win_prob"] = model.predict_proba(X_test)[:, 1]     # probability that each driver wins
test["pole_score"] = -test["grid"].fillna(99)            # baseline score: better (lower) grid = higher score

def race_winner_accuracy(frame, score_col):
    # For every race: pick the driver with the highest score, check if he really won
    hits = []                                            # 1 = correct race, 0 = wrong race
    for race_id, race in frame.groupby("race_id"):       # look at one race at a time
        pick = race.loc[race[score_col].idxmax()]        # the row (driver) with the highest score
        hits.append(int(pick["won"] == 1))               # 1 if that driver was the real winner
    return np.mean(hits)                                 # share of races we got right

model_acc    = race_winner_accuracy(test, "win_prob")    # our model
baseline_acc = race_winner_accuracy(test, "pole_score")  # "pole-sitter wins" baseline

print(f"Model    race-winner accuracy: {model_acc:.0%}")
print(f"Baseline race-winner accuracy: {baseline_acc:.0%}")

# race by race table: what did we predict vs what happened?
rows = []
for race_id, race in test.groupby("race_id"):
    predicted = race.loc[race["win_prob"].idxmax(), "driver_id"]       # our pick
    actual = race.loc[race["won"] == 1, "driver_id"]                    # the real winner
    rows.append({"race": race["race_name"].iloc[0],
                 "predicted": predicted,
                 "actual_winner": actual.iloc[0] if len(actual) else "?",
                 "correct": predicted == (actual.iloc[0] if len(actual) else None)})
pd.DataFrame(rows)

Model    race-winner accuracy: 83%
Baseline race-winner accuracy: 67%


,race,predicted,actual_winner,correct
0,Italian Grand Prix,antonelli,antonelli,True
1,Spanish Grand Prix,antonelli,antonelli,True
2,Azerbaijan Grand Prix,russell,russell,True
3,Bahrain Grand Prix in Malaysia,max_verstappen,max_verstappen,True
4,Singapore Grand Prix,max_verstappen,max_verstappen,True
5,United States Grand Prix,max_verstappen,antonelli,False


### What did the model learn? (Feature importance)

**What it does:** shows which columns the forest relied on most.

Good signs: grid position, team form and driver form near the top. If a strange column dominated, we would suspect a bug or leakage.

In [8]:
names = model.named_steps["prep"].get_feature_names_out()          # column names after preprocessing
importance = model.named_steps["forest"].feature_importances_      # how much each column helped the trees
imp = pd.Series(importance, index=names).sort_values(ascending=False)  # sort biggest first
imp.head(10)                                                       # top 10 most useful columns

,0
num__grid,0.253951
num__driver_points_5,0.188634
num__driver_avg_finish_5,0.150175
num__team_points_5,0.137293
num__driver_wins_5,0.079696
onehot__constructor_id_red_bull,0.045800
driver__driver_id,0.043327
onehot__constructor_id_ferrari,0.010442
onehot__constructor_id_mercedes,0.009452
onehot__constructor_id_aston_martin,0.006886


## Step 8: Predict the upcoming race

**What it does:**
1. Re-trains the model on **all** races so far (now including the 6 test races, because for a real prediction we want to use every bit of data).
2. Lists the drivers/teams from the most recent race (the current grid).
3. Looks up the qualifying grid for that race. If qualifying has not happened yet, the API returns nothing, the grid stays NaN and our imputer fills it with the median.
4. Builds the same form features from each driver's latest 5 races.
5. Prints each driver's chance of winning, scaled so all chances add to 100%.

**Why scale to 100%?** Exactly one driver wins, so the probabilities should sum to 1. The raw forest outputs do not, so we divide by the total.

In [9]:
final_model = clone(model)                               # fresh untrained copy of the same pipeline
final_model.fit(df[features], df["won"])                 # train on ALL completed races

nxt = get_json(f"{BASE}/current/next.json")["MRData"]["RaceTable"]["Races"]   # ask API for the next race
if not nxt:                                              # empty = season is over / no race scheduled
    print("No upcoming race found (season may be finished).")
else:
    race = nxt[0]                                        # the next race
    season, rnd = race["season"], int(race["round"])     # its year and round number
    circuit = race["Circuit"]["circuitId"]               # its track
    print("Upcoming race:", race["raceName"], "| round", rnd, "| date", race["date"])

    # qualifying grid
    qual = get_json(f"{BASE}/{season}/{rnd}/qualifying.json")["MRData"]["RaceTable"]["Races"]
    grid_map = {}                                        # driver -> starting slot
    if qual:                                             # qualifying has happened
        grid_map = {q["Driver"]["driverId"]: int(q["position"])
                    for q in qual[0]["QualifyingResults"]}
    print("Qualifying grid available:", bool(grid_map))

    # today's driver line-up = drivers of the latest completed race
    lineup = df[df["race_id"] == df["race_id"].max()][["driver_id", "constructor_id"]]

    new_rows = []
    for _, r in lineup.iterrows():                       # one row per driver
        driver_hist = df[df["driver_id"] == r["driver_id"]].tail(5)           # driver's last 5 races
        team_hist = team[team["constructor_id"] == r["constructor_id"]].tail(5)  # team's last 5 races
        new_rows.append({
            "grid": grid_map.get(r["driver_id"], np.nan),               # starting slot (NaN if unknown)
            "driver_avg_finish_5": driver_hist["position"].mean(),             # same features as training
            "driver_wins_5": driver_hist["won"].sum(),
            "driver_points_5": driver_hist["points"].mean(),
            "team_points_5": team_hist["team_points"].mean(),
            "driver_id": r["driver_id"],
            "constructor_id": r["constructor_id"],
            "circuit_id": circuit,
        })
    upcoming = pd.DataFrame(new_rows)                    # table of the upcoming race

    upcoming["win_prob"] = final_model.predict_proba(upcoming[features])[:, 1]   # raw win chance
    upcoming["win_prob"] = upcoming["win_prob"] / upcoming["win_prob"].sum()      # make them add up to 100%
    result = upcoming.sort_values("win_prob", ascending=False)[["driver_id", "constructor_id", "grid", "win_prob"]]
    result["win_prob"] = (result["win_prob"] * 100).round(1)                      # show as percent
    print("\nPredicted winner:", result.iloc[0]["driver_id"])
    display(result.head(5).rename(columns={"win_prob": "win_chance_%"}))

Upcoming race: Mexico City Grand Prix | round 19 | date 2026-11-01
Qualifying grid available: False

Predicted winner: max_verstappen


,driver_id,constructor_id,grid,win_chance_%
2,max_verstappen,red_bull,NaN,36.6
0,antonelli,mercedes,NaN,24.4
5,hadjar,red_bull,NaN,11.3
1,leclerc,ferrari,NaN,9.0
7,russell,mercedes,NaN,5.2


## Conclusion

- ML workflow: download -> clean -> create features -> time-based split -> pipeline (SimpleImputer + OrdinalEncoder + OneHotEncoder + Random Forest) -> evaluate -> predict.
- We chose race-winner accuracy over MAE, because MAE (and plain accuracy) reward a model that says "nobody wins" and does not rank drivers inside a race. Winner accuracy answers real question and is compared against the simple "pole-sitter wins" baseline.
- With only 6 test races, one race changes the result by ~17 points. If the model does not clearly beat the pole baseline, that is normal, since a lot of the winner is already decided by the starting grid and the car.

  Model race-winner accuracy: 83%

  Baseline race-winner accuracy: 67%

- The 2026 season has new regulations and some new teams/line-ups, so older seasons are less reliable for predicting 2026. The 'OneHotEncoder(handle_unknown="ignore")' and 'OrdinalEncoder(unknown_value=-1)' settings keep the model from crashing on new teams and drivers.
- The prediction is a probability. Rain, safety cars and strategy make F1 random.

## What to improve in next version
1. Using walk-forward validation so the score comes from 30+ races instead of 6.
2. Features: qualifying lap-time gap to pole, long-run pace from practice, driver's past results at specific track, team reliability (DNF rate), pit-stop performance, weather/rain, championship points position.
3. 2026 race should count more than a 2021 race because the cars/regulations changed.
4. Using XGBoost and tune settings with time-series cross-validation.
5. Using a "learning-to-rank" model or predict finishing position and sort drivers.
6. Adding top-3 accuracy and log-loss (it scores the quality of the probabilities themselves).